[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/STAT3009_MF.ipynb)

# Notebook 05: Matrix factorization (MF)

**STAT3009 · Recommender Systems**  
**Suggested time:** 105–120 minutes of guided coding and discussion

ML II selected model settings with cross-validation. Here we keep that workflow, but replace the three-mean predictor with **matrix factorization**: a baseline plus a learned user–item interaction.

By the end, you should be able to:

1. explain why user and item *latent factors* are learned from ratings rather than observed columns;
2. compute a prediction from $\mu+a_u+b_i+p_u^\mathsf{T}q_i$;
3. translate one item or user update into a familiar Ridge regression;
4. read an MF estimator's `fit` and `predict` methods, then select $K$ and $\lambda$ with `GridSearchCV`;
5. explain how predictions fall back when a user or item was unseen during fitting.

The slides use a **1–5 toy example** for intuition. The real classroom dataset below uses **binary ratings (0/1)**, as in ML I and ML II. Do not compare their numeric rating scales.

## 1. Setup and the observed rating rows

We use the same fixed `b-rs` split as the earlier notebooks. `train.csv` supplies the observed pairs for fitting and cross-validation. We will open `test.csv` only at the end.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.utils.validation import check_is_fitted

pd.set_option('display.max_rows', 12)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

RS_BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/b-rs'
)
rs_train = pd.read_csv(f'{RS_BASE_URL}/train.csv')
display(rs_train[['user_id', 'movie_id', 'rating']].head())

In [ ]:
X_train_rs = rs_train[['user_id', 'movie_id']].to_numpy(dtype=int)
y_train_rs = rs_train['rating'].to_numpy(dtype=float)

print('X shape:', X_train_rs.shape, '— columns are [user ID, item ID]')
print('y shape:', y_train_rs.shape, '— observed ratings')
print('number of observed ratings:', len(y_train_rs))
print('number of users:', len(set(X_train_rs[:, 0])))
print('number of items:', len(set(X_train_rs[:, 1])))
print('possible user–item pairs are much more numerous than observed rows.')

**Only observed rows enter the loss.** A missing user–item pair is *not* a rating of zero. Here `0` is an actual observed binary rating; a missing pair has no row in `rs_train`.

## 2. Why add an interaction?

A baseline of the form $\mu+a_u+b_i$ can describe a generally generous user and a generally popular item. It cannot express a **reversal** where User 0 prefers Item 0 but User 1 prefers Item 1: for both users, the baseline difference between Item 0 and Item 1 is always $b_0-b_1$.

The slides use the following **hand-picked teaching values**. They are not columns in the dataset and were not estimated from the four toy ratings. In a real fit, the model learns the profile vectors from observed ratings.

In [ ]:
toy_mu = 3.0
toy_user_factors = {0: np.array([1.0, 0.2]), 1: np.array([0.2, 1.0])}
toy_item_factors = {0: np.array([1.5, -1.5]), 1: np.array([-1.5, 1.5])}

for user in [0, 1]:
    for item in [0, 1]:
        interaction = toy_user_factors[user] @ toy_item_factors[item]
        prediction = toy_mu + interaction  # toy user/item biases are zero
        print(f'User {user}, Item {item}: baseline {toy_mu:.1f} '
              f'+ interaction {interaction:+.1f} = {prediction:.1f}')

### Quick check

For **User 1, Item 0**, which two coordinates are multiplied, and why does the interaction *lower* the prediction? If both item biases were nonzero, where would you add $b_0$?

The illustrative labels “action-like” and “romance-like” help us picture the toy vectors. **Actual latent coordinates do not come with observed genre labels.** The only inputs to `predict` will be the two IDs; the fitted model looks up its stored parameters.

## 3. MF through our four questions

| Question | MF answer |
|---|---|
| **Model** | $\widehat r_{ui}=\mu+a_u+b_i+p_u^\mathsf{T}q_i$ for a user–item pair $(u,i)$ |
| **Hyperparameters** | $K$: number of latent coordinates; $\lambda$: regularization strength. We choose these before fitting. |
| **Loss / fitting criterion** | Sum of squared errors on **observed training pairs**, plus $\lambda$ times the squared sizes of the latent factors. |
| **Learned parameters** | $\mu$ from the fitting-fold mean; user/item biases $a,b$ and factor vectors $P,Q$ learned from fitting-fold ratings. |

With $\Omega_{\mathrm{train}}$ denoting the observed fitting pairs, our exact convention is

$$
\min_{a,b,P,Q} J
=\sum_{(u,i)\in\Omega_{\mathrm{train}}}
  (r_{ui}-\mu-a_u-b_i-p_u^\mathsf{T}q_i)^2
 +\lambda\bigl(\|P\|_F^2+\|Q\|_F^2\bigr),
\qquad
\mu=\operatorname{mean}(r_{ui}\text{ in the fitting fold}).
$$

The **fitting criterion** is a sum plus a penalty; the **reporting metric** below is RMSE. The global mean stays fixed during ALS. The user and item biases are unpenalized Ridge intercepts; only $P,Q$ receive the $\lambda$ penalty. `n_iters` and `random_state` will also be constructor settings for the algorithm, but we focus model selection on $K$ and $\lambda$.

In [ ]:
mu_demo = y_train_rs.mean()
print(f'global mean from training rows only: {mu_demo:.4f}')
print('The user/item factors are absent from rs_train; fit must estimate them.')

## 4. One ALS update is one Ridge regression

The model is awkward to fit jointly because $p_u^\mathsf{T}q_i$ contains two unknown vectors. **Alternating least squares (ALS)** fixes the user side while updating the item side, then fixes the item side while updating the user side.

For one item $i$, assume the current user biases $a_u$ and factors $p_u$ are fixed. Its new item bias $b_i$ and factor vector $q_i$ solve

$$
\min_{b,q}\sum_{u\in U_i}
\bigl[(r_{ui}-\mu-a_u)-(b+p_u^\mathsf{T}q)\bigr]^2
+\lambda\|q\|_2^2.
$$

So the Ridge features are $p_u^\mathsf{T}$, the targets are $r_{ui}-\mu-a_u$, and `fit_intercept=True` returns the item bias in `intercept_` and its factors in `coef_`. We subtract $\mu$ from the target because it is fixed, rather than asking Ridge to learn a second global mean.

In [ ]:
# Teaching state before the first item update: zero biases, small random factors.
K_demo = 2
lambda_demo = 1.0
rng = np.random.default_rng(42)
n_users_demo = X_train_rs[:, 0].max() + 1
n_items_demo = X_train_rs[:, 1].max() + 1
a_demo = np.zeros(n_users_demo)
b_demo = np.zeros(n_items_demo)
P_demo = rng.normal(0, 0.1, size=(n_users_demo, K_demo))
Q_demo = np.zeros((n_items_demo, K_demo))

# Select the most-rated item so the small Ridge example has many rows.
item_i = int(rs_train['movie_id'].value_counts().index[0])
item_rows = np.where(X_train_rs[:, 1] == item_i)[0]
user_ids_for_item = X_train_rs[item_rows, 0]

X_item = P_demo[user_ids_for_item]
y_item = y_train_rs[item_rows] - mu_demo - a_demo[user_ids_for_item]

item_ridge = Ridge(alpha=lambda_demo, fit_intercept=True)
item_ridge.fit(X_item, y_item)
b_demo[item_i] = item_ridge.intercept_
Q_demo[item_i] = item_ridge.coef_

print('Ridge feature shape:', X_item.shape, '= two factor columns')
print('item bias b_i:', b_demo[item_i])
print('item factors q_i:', Q_demo[item_i])

### In-class practice: reverse the roles

Choose a user $u$ who rated the item just updated. Keep the current item bias/factors fixed. Complete the **user** Ridge features, targets, and coefficient assignment. We only updated one item's $q_i$ above; all other item vectors are currently zero. That is fine for seeing the mechanics of one update.

```python
user_u = int(user_ids_for_item[0])
user_rows = np.where(X_train_rs[:, 0] == user_u)[0]
item_ids_for_user = X_train_rs[user_rows, 1]

# X_user = ...
# y_user = y_train_rs[user_rows] - ... - ...
# user_ridge = Ridge(alpha=lambda_demo, fit_intercept=True).fit(X_user, y_user)
# a_demo[user_u] = ...
# P_demo[user_u] = ...
```

Try the four missing expressions before opening the solution cell.

### Solution

In [ ]:
user_u = int(user_ids_for_item[0])
user_rows = np.where(X_train_rs[:, 0] == user_u)[0]
item_ids_for_user = X_train_rs[user_rows, 1]

X_user = Q_demo[item_ids_for_user]
y_user = y_train_rs[user_rows] - mu_demo - b_demo[item_ids_for_user]

user_ridge = Ridge(alpha=lambda_demo, fit_intercept=True)
user_ridge.fit(X_user, y_user)
a_demo[user_u] = user_ridge.intercept_
P_demo[user_u] = user_ridge.coef_

print('Ridge feature shape:', X_user.shape)
print('user bias a_u:', a_demo[user_u])
print('user factors p_u:', P_demo[user_u])

**Discuss:** Which values were fixed during the item update? Which were fixed during the user update? Why is a single update of one item and one user **not** a complete MF fit?

## 5. Put the repeated updates in an sklearn estimator

The supplied class performs the same Ridge calculation for **every observed item**, then for **every observed user**, and repeats. Read the three marked parts: constructor choices, `fit`, and `predict`. You do not need to write this whole class from memory; your coding task is to understand and use its pieces.

For an unseen user or item, the corresponding bias and interaction cannot be looked up. `predict` falls back to the known terms: $\mu+b_i$, $\mu+a_u$, or just $\mu$. This is the same idea as the three-mean notebook.

In [ ]:
class BiasedMF(RegressorMixin, BaseEstimator):
    def __init__(self, K=2, lambda_=1.0, n_iters=3, random_state=42):
        # Settings chosen before fit; GridSearchCV can read and change them.
        self.K = K
        self.lambda_ = lambda_
        self.n_iters = n_iters
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=int)
        y = np.asarray(y, dtype=float)
        if X.ndim != 2 or X.shape[1] != 2 or len(X) != len(y) or len(y) == 0:
            raise ValueError('X must have two ID columns and match a nonempty y.')
        if np.any(X < 0):
            raise ValueError('User and item IDs must be nonnegative integers.')
        if self.K < 1 or self.lambda_ < 0 or self.n_iters < 1:
            raise ValueError('Use K >= 1, lambda_ >= 0, and n_iters >= 1.')

        users, items = X[:, 0], X[:, 1]
        n_users, n_items = users.max() + 1, items.max() + 1
        self.mu_ = y.mean()  # recomputed from this fit's rows, including in CV
        self.a_ = np.zeros(n_users)
        self.b_ = np.zeros(n_items)
        rng = np.random.default_rng(self.random_state)
        self.P_ = rng.normal(0, 0.1, size=(n_users, self.K))
        self.Q_ = rng.normal(0, 0.1, size=(n_items, self.K))

        rows_by_user = [[] for _ in range(n_users)]
        rows_by_item = [[] for _ in range(n_items)]
        for row, (u, i) in enumerate(X):
            rows_by_user[u].append(row)
            rows_by_item[i].append(row)

        self.seen_users_ = np.array([bool(rows) for rows in rows_by_user])
        self.seen_items_ = np.array([bool(rows) for rows in rows_by_item])
        self.P_[~self.seen_users_] = 0.0
        self.Q_[~self.seen_items_] = 0.0
        self.objective_history_ = [self.objective(X, y)]

        for _ in range(self.n_iters):
            # Fix users, update each observed item: intercept = b_i, coefficients = q_i.
            for i, rows in enumerate(rows_by_item):
                if not rows:
                    continue
                user_ids = users[rows]
                features = self.P_[user_ids]
                targets = y[rows] - self.mu_ - self.a_[user_ids]
                ridge = Ridge(alpha=self.lambda_, fit_intercept=True)
                ridge.fit(features, targets)
                self.b_[i] = ridge.intercept_
                self.Q_[i] = ridge.coef_

            # Fix items, update each observed user: intercept = a_u, coefficients = p_u.
            for u, rows in enumerate(rows_by_user):
                if not rows:
                    continue
                item_ids = items[rows]
                features = self.Q_[item_ids]
                targets = y[rows] - self.mu_ - self.b_[item_ids]
                ridge = Ridge(alpha=self.lambda_, fit_intercept=True)
                ridge.fit(features, targets)
                self.a_[u] = ridge.intercept_
                self.P_[u] = ridge.coef_

            self.objective_history_.append(self.objective(X, y))
        return self

    def predict(self, X):
        check_is_fitted(self, 'mu_')
        X = np.asarray(X, dtype=int)
        if X.ndim != 2 or X.shape[1] != 2:
            raise ValueError('X must have two columns: [user ID, item ID].')
        users, items = X[:, 0], X[:, 1]
        known_user = (users >= 0) & (users < len(self.seen_users_))
        known_item = (items >= 0) & (items < len(self.seen_items_))
        known_user[known_user] = self.seen_users_[users[known_user]]
        known_item[known_item] = self.seen_items_[items[known_item]]

        predictions = np.full(len(X), self.mu_)
        predictions[known_user] += self.a_[users[known_user]]
        predictions[known_item] += self.b_[items[known_item]]
        both = known_user & known_item
        predictions[both] += np.sum(
            self.P_[users[both]] * self.Q_[items[both]], axis=1
        )
        return predictions

    def objective(self, X, y):
        # Same sum-loss convention as the slides; only fitted rating rows enter.
        residuals = np.asarray(y, dtype=float) - self.predict(X)
        penalty = (
            np.sum(self.P_ ** 2) + np.sum(self.Q_ ** 2)
        )
        return np.sum(residuals ** 2) + self.lambda_ * penalty

### Fit one model and inspect what it learned

The constructor stores settings; `fit` creates learned attributes ending in `_`. The objective should go down (or remain the same) after complete ALS iterations because each Ridge update minimizes its own block of the **same** objective. The problem is not jointly convex, so this does not promise a globally best MF model.

In [ ]:
mf_demo = BiasedMF(K=2, lambda_=1.0, n_iters=3, random_state=42)
mf_demo.fit(X_train_rs, y_train_rs)

print('settings:', mf_demo.get_params())
print('learned global mean:', mf_demo.mu_)
print('user factor shape:', mf_demo.P_.shape)
print('item factor shape:', mf_demo.Q_.shape)
display(pd.DataFrame({
    'completed ALS iterations': range(len(mf_demo.objective_history_)),
    'training objective': mf_demo.objective_history_,
}))
print('training RMSE:', root_mean_squared_error(
    y_train_rs, mf_demo.predict(X_train_rs)
))

**Discuss:** `training objective` includes the penalty, whereas `training RMSE` does not. Neither number tells us which $K,\lambda$ will generalize best. We need validation folds.

## 6. Choose $K$ and $\lambda$ with cross-validation

As in ML II, `GridSearchCV` uses the estimator interface to call `fit` and `predict` repeatedly. Each candidate uses the **same three folds of observed rating rows**. Each fold learns its own $\mu,a,b,P,Q$ from its fitting rows. The validation ratings only score the predictions; they do not enter that fold's fit.

A few validation rows may contain an ID absent from that fold's fitting rows. The fallback rule in `predict` handles those rows, so this CV score includes both familiar and cold-start pairs.

### In-class practice

Construct `mf_search` to compare `K = 2, 5` and `lambda_ = 0.1, 10.0`, using three-fold validation RMSE. Which two names belong in the parameter grid? Why are `mu_` and `P_` **not** in that grid?

```python
# mf_search = GridSearchCV(
#     estimator=BiasedMF(n_iters=3, random_state=42),
#     param_grid={ ... },
#     scoring='neg_root_mean_squared_error',
#     cv=3,
#     refit=True,
# )
# mf_search.fit(X_train_rs, y_train_rs)
```

The negative score is sklearn's convention for a metric that should be **minimized**. Display positive RMSE by changing its sign.

### Solution

In [ ]:
mf_search = GridSearchCV(
    estimator=BiasedMF(n_iters=3, random_state=42),
    param_grid={'K': [2, 5], 'lambda_': [0.1, 10.0]},
    scoring='neg_root_mean_squared_error',
    cv=3,
    refit=True,
)
mf_search.fit(X_train_rs, y_train_rs)

cv_table = pd.DataFrame(mf_search.cv_results_)[
    ['param_K', 'param_lambda_', 'mean_test_score', 'rank_test_score']
].copy()
cv_table['validation_RMSE'] = -cv_table.pop('mean_test_score')
display(cv_table.sort_values('validation_RMSE'))
print('selected settings:', mf_search.best_params_)
print(f'three-fold validation RMSE: {-mf_search.best_score_:.4f}')

**Discuss:** Did changing $K$ or $\lambda$ matter more for these candidates? The table estimates performance on held-out **training** folds. It is not a test-set leaderboard. `refit=True` has already refitted `best_estimator_` on **all** training rows after selection.

## 7. What if a user or item is new?

The model has learned vectors only for IDs observed in its fit. For a new user, it has no $a_u,p_u$; for a new item, it has no $b_i,q_i$. The prediction rule uses the available terms:

| Pair | Prediction |
|---|---|
| Known user, known item | $\mu+a_u+b_i+p_u^\mathsf{T}q_i$ |
| New user, known item | $\mu+b_i$ |
| Known user, new item | $\mu+a_u$ |
| New user, new item | $\mu$ |

An ID can be absent from a CV fitting fold even when it is within the dataset's overall numeric ID range. That is why `seen_users_` and `seen_items_` store **membership in this fit**, not just the array lengths.

In [ ]:
final_mf = mf_search.best_estimator_
known_user_id, known_item_id = X_train_rs[0]
new_user_id = len(final_mf.seen_users_)
new_item_id = len(final_mf.seen_items_)

four_pairs = np.array([
    [known_user_id, known_item_id],
    [new_user_id, known_item_id],
    [known_user_id, new_item_id],
    [new_user_id, new_item_id],
])
display(pd.DataFrame({
    'case': ['known + known', 'new user', 'new item', 'both new'],
    'user_id': four_pairs[:, 0],
    'item_id': four_pairs[:, 1],
    'prediction': final_mf.predict(four_pairs),
}))

### Quick code check

Use `final_mf.mu_`, `final_mf.a_`, and `final_mf.b_` to calculate the **new-user/known-item** and **known-user/new-item** predictions by hand. Compare them to the second and third rows above.

In [ ]:
print('new user + known item:', final_mf.mu_ + final_mf.b_[known_item_id])
print('known user + new item:', final_mf.mu_ + final_mf.a_[known_user_id])

## 8. Final classroom evaluation

Only now do we open the labeled `test.csv`. These labels are available for teaching; in a real prediction task they would be hidden. We use the selected, already-refitted model **once** to report final performance. The cold-start breakdown helps us see which cases cannot use an interaction term.

In [ ]:
rs_test = pd.read_csv(f'{RS_BASE_URL}/test.csv')
X_test_rs = rs_test[['user_id', 'movie_id']].to_numpy(dtype=int)
y_test_rs_demo = rs_test['rating'].to_numpy(dtype=float)
test_predictions = final_mf.predict(X_test_rs)

known_user = np.array([
    0 <= u < len(final_mf.seen_users_) and final_mf.seen_users_[u]
    for u in X_test_rs[:, 0]
])
known_item = np.array([
    0 <= i < len(final_mf.seen_items_) and final_mf.seen_items_[i]
    for i in X_test_rs[:, 1]
])

cases = {
    'overall': np.ones(len(X_test_rs), dtype=bool),
    'known user + known item': known_user & known_item,
    'new user or new item': ~(known_user & known_item),
}
report = []
for name, mask in cases.items():
    report.append({
        'case': name,
        'rows': int(mask.sum()),
        'test_RMSE': root_mean_squared_error(
            y_test_rs_demo[mask], test_predictions[mask]
        ) if mask.any() else np.nan,
    })
display(pd.DataFrame(report))

**Interpretation:** The cold-start number combines pairs with a new user, a new item, or both. It may use different terms from the full MF rule. These test results are a final report, **not** a reason to rerun the grid search with different candidates.

## Exit ticket

1. In the toy example, were $p_0$ and $q_0$ observed features or illustrative choices? Where do real MF profiles come from?
2. What are the inputs and targets of the Ridge problem for a single item update? What do `intercept_` and `coef_` represent?
3. What is chosen before `fit`, and what is recomputed inside each CV fitting fold?
4. For a new user and a known item, which learned terms can still be used?